In [ ]:

import os
import sys
import time
from pathlib import Path
from datetime import datetime

#3rd party imports
import requests
import pandas as pd
from dotenv import load_dotenv
from stem import Signal
from stem.control import Controller
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.action_chains import ActionChains



ModuleNotFoundError: No module named 'dotenv'

In [ ]:
# --- Configuration and Setup ---
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
ruta_env = Path(__file__).resolve().parent.parent.parent / ".env"
load_dotenv(dotenv_path=ruta_env)

# --- Selenium WebDriver Utilities ---
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.action_chains import ActionChains

def get_driver():
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--proxy-server=socks5://")
    try:
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)
    except Exception as e:
        return e
    return driver

def foward_button(driver):
    the = driver.find_elements(By.CLASS_NAME, "show-more")[-1]
    the.click()
    return driver.current_url.split("cursor=")[-1]

# --- Nitter Scraping Functions ---
def find_nitter(driver, date="", cursor=""):
    bucle = True
    if date == "":
        url = f"https://nitter.net/_minecogob?cursor={cursor}"
    else:
        url = f"https://nitter.net/search?f=tweets&q=from%3A_minecogob+until%3A{date}&since=&until=&min_faves= "
    print(url)
    attempts = 0
    while bucle:
        try:
            driver.get(url)
            bucle = False
        except:
            attempts += 1
            if attempts >= 4:
                renew_tor_ip()
                attempts = 0
        time.sleep(2)

def load_page(driver, csv_file="gobierno_economia.csv"):
    tweets_data = []
    the = driver.find_elements(By.CLASS_NAME, "show-more")[-1]
    actions = ActionChains(driver)
    actions.move_to_element(the).perform()
    time.sleep(1)
    actions = ActionChains(driver)
    actions.move_to_element(the).perform()
    time.sleep(1)
    tweets = driver.find_elements(By.CLASS_NAME, "timeline-item")
    iso_date = ""
    for tweet in tweets:
        try:
            element_date = tweet.find_element(By.CLASS_NAME, "tweet-date").find_element(By.TAG_NAME, "a")
            date = element_date.get_attribute("title")
            hashtag_elements = tweet.find_elements(By.XPATH, ".//div[contains(@class, 'tweet-content')]//a[contains(@href, 'q=%23')]")
            hashtags = [ht.text for ht in hashtag_elements if ht.text.startswith("#")]
            username = tweet.find_element(By.CLASS_NAME, "username").text
            content = tweet.find_element(By.CLASS_NAME, "tweet-content").text
            attachments = tweet.find_elements(By.CLASS_NAME, "attachment")
            type_content = "Texto simple"
            url_media = None
            if attachments:
                if tweet.find_elements(By.CLASS_NAME, "still-image"):
                    type_content = "Imagen"
                    url_media = tweet.find_element(By.CLASS_NAME, "still-image").get_attribute("href")
                elif tweet.find_elements(By.CLASS_NAME, "video-container") or tweet.find_elements(By.CLASS_NAME, "media-gif"):
                    type_content = "Video/GIF"
            stats_elements = tweet.find_elements(By.CLASS_NAME, "tweet-stat")
            comments = stats_elements[0].text.strip() or "0"
            retweets = stats_elements[1].text.strip() or "0"
            likes = stats_elements[2].text.strip() or "0"
            views = stats_elements[3].text.strip() or "0"
            tweet_id = tweet.find_element(By.TAG_NAME, "a").get_attribute("href").split("/")[-1].split("#")[0]
            tweet_data = {
                "id": tweet_id,
                "date": date,
                "username": username,
                "content": content,
                "hashtags": ", ".join(hashtags),
                "type": type_content,
                "url_media": url_media,
                "comments": comments,
                "retweets": retweets,
                "likes": likes,
                "views": views
            }
            tweets_data.append(tweet_data)
            fecha_limpia = date.replace("·", "").strip()
            objeto_fecha = datetime.strptime(fecha_limpia, "%b %d, %Y %I:%M %p UTC")
            iso_date = objeto_fecha.strftime("%Y-%m-%d")
        except Exception as e:
            print(f"Error en la extracción: {e}")
    df = pd.DataFrame(tweets_data)
    df.to_csv(csv_file, mode='a', index=False, header=not os.path.exists(csv_file))
    return iso_date

# --- Tor Utilities ---
from stem import Signal
from stem.control import Controller

def renew_tor_ip():
    with Controller.from_port(port=9051) as controller:
        controller.authenticate()
        controller.signal(Signal.NEWNYM)
        print("We asked Tor for a new IP..")

# --- Main Scraping Loop ---
def main():
    continue_scraping = True
    cursor = ""
    date = "2029-01-01"
    number_pages = 0
    driver = get_driver()
    find_nitter(driver, cursor="")
    while continue_scraping:
        try:
            prior_date = load_page(driver)
            cursor = foward_button(driver)
            if datetime.fromisoformat(date) >= datetime.fromisoformat(prior_date):
                with open("cursores.txt", "a") as w:
                    w.write(cursor + "\n")
                with open("fechas.txt", "a") as w:
                    w.write(date + "\n")
                number_pages += 1
                date = prior_date
            else:
                renew_tor_ip()
                time.sleep(2)
                find_nitter(driver, date)
        except Exception as e:
            print(f"Error in main loop: {e}")
            find_nitter(driver, date)
        time.sleep(3)

# main()
